# ML Churn Classification — Financial Services Demo
Train and evaluate classification models to predict customer churn using features from `CONSUMPTION.DT_CHURN_FEATURES`.

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from snowflake.snowpark import Session
import os

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)

In [ ]:
# Cell 2: Connect to Snowflake
connection_params = {
    'account':   os.environ.get('SNOWFLAKE_ACCOUNT', 'your_account'),
    'user':      os.environ.get('SNOWFLAKE_USER', 'your_user'),
    'password':  os.environ.get('SNOWFLAKE_PASSWORD', ''),
    'role':      'ACCOUNTADMIN',
    'warehouse': 'FINSERV_WH',
    'database':  'FINSERV_DB',
    'schema':    'CONSUMPTION'
}

session = Session.builder.configs(connection_params).create()
print(f'Connected: {session.get_current_database()}.{session.get_current_schema()}')

In [ ]:
# Cell 3: Load churn features
df_sp = session.table('DT_CHURN_FEATURES')
print(f'Schema: {df_sp.schema}')
print(f'Row count: {df_sp.count():,}')

df = df_sp.to_pandas()
df.head()

In [ ]:
# Cell 4: EDA — Target distribution
print('Churn distribution:')
print(df['IS_CHURNED'].value_counts())
print(f'Churn rate: {df["IS_CHURNED"].mean():.2%}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
df['IS_CHURNED'].value_counts().plot.bar(ax=axes[0], color=['steelblue', 'coral'])
axes[0].set_title('Churn Distribution')
axes[0].set_xticklabels(['Active', 'Churned'], rotation=0)

# Numeric feature distributions
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols = [c for c in numeric_cols if c not in ['CUSTOMER_ID', 'IS_CHURNED']]
df[numeric_cols[:6]].hist(bins=30, ax=axes[1] if len(numeric_cols) == 1 else None, figsize=(14, 8))
plt.suptitle('Feature Distributions')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: Correlation matrix
corr_cols = [c for c in numeric_cols if c != 'CUSTOMER_ID'] + ['IS_CHURNED']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Feature engineering
feature_cols = [c for c in numeric_cols if c != 'CUSTOMER_ID']
target_col = 'IS_CHURNED'

print(f'Features ({len(feature_cols)}): {feature_cols}')
print(f'Target: {target_col}')

X = df[feature_cols].fillna(0)
y = df[target_col].astype(int)

print(f'X shape: {X.shape}, y shape: {y.shape}')

In [ ]:
# Cell 7: Train/test split
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'Train: {X_train.shape[0]:,} samples')
print(f'Test:  {X_test.shape[0]:,} samples')
print(f'Train churn rate: {y_train.mean():.2%}')
print(f'Test churn rate:  {y_test.mean():.2%}')

In [ ]:
# Cell 8: Train 3 models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    'LogisticRegression': LogisticRegression(max_iter=1000, random_state=42),
    'RandomForest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

trained = {}
for name, model in models.items():
    data = X_train_scaled if name == 'LogisticRegression' else X_train
    model.fit(data, y_train)
    trained[name] = model
    print(f'{name} trained.')

In [ ]:
# Cell 9: Evaluate models
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report
)

results = []
for name, model in trained.items():
    data = X_test_scaled if name == 'LogisticRegression' else X_test
    y_pred = model.predict(data)
    y_prob = model.predict_proba(data)[:, 1]

    metrics = {
        'Model':     name,
        'Accuracy':  accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall':    recall_score(y_test, y_pred, zero_division=0),
        'F1':        f1_score(y_test, y_pred, zero_division=0),
        'AUC-ROC':   roc_auc_score(y_test, y_prob),
    }
    results.append(metrics)
    print(f'\n=== {name} ===')
    print(classification_report(y_test, y_pred, zero_division=0))

df_results = pd.DataFrame(results)
df_results

In [ ]:
# Cell 10: Confusion matrices
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (name, model) in zip(axes, trained.items()):
    data = X_test_scaled if name == 'LogisticRegression' else X_test
    y_pred = model.predict(data)
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=['Active', 'Churned']).plot(ax=ax, cmap='Blues')
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 11: ROC curves
from sklearn.metrics import roc_curve

fig, ax = plt.subplots(figsize=(8, 6))
for name, model in trained.items():
    data = X_test_scaled if name == 'LogisticRegression' else X_test
    y_prob = model.predict_proba(data)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    ax.plot(fpr, tpr, label=f'{name} (AUC={auc:.3f})')

ax.plot([0,1], [0,1], 'k--', alpha=0.3)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — Churn Classification')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cell 12: Feature importance (best tree model)
best_tree = 'GradientBoosting'
importances = trained[best_tree].feature_importances_
feat_imp = pd.DataFrame({'Feature': feature_cols, 'Importance': importances})
feat_imp = feat_imp.sort_values('Importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(feat_imp['Feature'], feat_imp['Importance'], color='steelblue')
ax.set_title(f'Feature Importance — {best_tree}')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 13: Register best model to Snowflake Model Registry
from snowflake.ml.registry import Registry

# Pick best model by AUC
best_idx = df_results['AUC-ROC'].idxmax()
best_name = df_results.loc[best_idx, 'Model']
best_model = trained[best_name]
print(f'Best model: {best_name} (AUC={df_results.loc[best_idx, "AUC-ROC"]:.4f})')

# Register
registry = Registry(session=session)
mv = registry.log_model(
    model=best_model,
    model_name='CHURN_CLASSIFIER',
    version_name='v1',
    sample_input_data=X_train.head(10),
    comment=f'Churn classification — {best_name}'
)
print(f'Registered: CHURN_CLASSIFIER v1')

In [ ]:
# Cell 14: Test inference from registry
mv_loaded = registry.get_model('CHURN_CLASSIFIER').version('v1')
predictions = mv_loaded.run(X_test.head(5), function_name='predict')
print('Sample predictions from registry:')
predictions

In [ ]:
# Cell 15: Model comparison summary
fig, ax = plt.subplots(figsize=(10, 5))
df_melt = df_results.melt(id_vars='Model', var_name='Metric', value_name='Score')
sns.barplot(data=df_melt, x='Metric', y='Score', hue='Model', ax=ax)
ax.set_title('Model Comparison — All Metrics')
ax.set_ylim(0, 1.05)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 16: Cleanup
session.close()
print('Session closed. Churn classification complete.')